<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main/Sankey_02/sankey2_master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sankey diagram 6
### Target Class → Land Use

#Import libaries

In [1]:
import requests
import pandas as pd
import plotly.graph_objects as go
import duckdb
import string
import numpy as np
import plotly.colors

#Adgang til Airtable

In [2]:
# Airtable credentials
api_key = 'patwjsizhgQyQkZkT.f9e8b1595df5b527d0d01d3a45af0dfa77eab63707e18398ad62f1f3818a9ce9'
base_id = 'apprKfEKZ2Ju74g9w'

In [3]:
# henter data fra Airtable via API og gemmer det i en pandas.DataFrame
def fetch_airtable_data(table_id):
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {api_key}"}
    records = []
    params = {}

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data: {response.text}")
        data = response.json()
        records.extend([record["fields"] | {"id": record["id"]} for record in data["records"]])
        if "offset" in data:
            params["offset"] = data["offset"]
        else:
            break

    return pd.DataFrame(records)

In [ ]:
# Fetch data

tabel0 = fetch_airtable_data("tblzHR1WHYHA5MlwQ")  # Policy Source
tabel1 = fetch_airtable_data("tblGwx3sWdVfvlMto")  # Target Class
tabel2 = fetch_airtable_data("tbl7OYOXduME11uh7")  # Targets (mellemtabel)
tabel3 = fetch_airtable_data("tblTRyuT48bBN24QG")  # Slut-land uses


In [48]:
# Fjern rækker med manglende værdier i specifikke kolonner

# For tabel0: behold kun rækker hvor 'Policy source' og'Targets (policy targets)' er udfyldt
tabel0 = tabel0.dropna(subset=['Name','Targets'])

# For tabel1: behold kun rækker hvor 'Target Class' og 'Targets' er udfyldt
tabel1 = tabel1.dropna(subset=['Name', 'Targets'])

# For tabel2: behold kun rækker hvor 'Targets', Target Class' og 'Land uses' er udfyldt
tabel2 = tabel2.dropna(subset=['Target name','Target Class', 'Land uses'])

# For tabel3: behold kun rækker hvor 'Targets' er udfyldt
tabel3 = tabel3.dropna(subset=['Name','Targets'])


In [56]:
# Explode og rename for videre behandling

# t0 = Policy Source → Targets
t0 = tabel0.copy().explode("Targets").rename(columns={
    "Targets": "target_id",
    "id": "policy_source_id",
    "Name": "policy_source_name"
})

# t1 = Target Class → Targets
t1 = tabel1.copy().explode("Targets").rename(columns={
    "Targets": "target_id",
    "Name": "target_class_name",
    "id": "target_class_id"
})

# t2 = Targets → Land Uses
t2 = tabel2.copy().explode("Land uses").rename(columns={
    "Land uses": "land_use_id",
    "Target name": "target_name",
    "id": "target_id"
})

# t3 = Land Uses (navne til land_use_id)
t3 = tabel3.copy().rename(columns={
    "Name": "land_use_name",
    "id": "land_use_id"
})



In [58]:
filtered_target_ids = t0["target_id"].unique()
t1 = t1[t1["target_id"].isin(filtered_target_ids)]
t2 = t2[t2["target_id"].isin(filtered_target_ids)]


In [50]:
# Filtrér direkte før registrering
allowed_policies = [
    "European Green Deal",
    "Common Agricultural Policy - Strategic plan 2023-2027",
    "EU biodiversity strategy 2030",
    "Aftale om et grønt Danmark (grøn trepart)",
    "Mere, bedre og større natur i Danmark",
    "Vandområdeplaner 2021-2027",
    "Technical Summary: Land Use and Climate Change",
    "Convention on Biological Diversity",
    "Common approach to integrating biodiversity and nature-based solutions for sustainable development into the United Nations policy and programme planning and delivery"
 ]
t0 = t0[t0["policy_source_name"].isin(allowed_policies)]

# allowed_groups = ["Climate", "Biodiversity", "Groundwater and water"]
# t2 = t2[t2["target_group_id"].isin(allowed_groups)]

In [29]:
#"filtrer til kun at indeholde Targets, der stammer fra de Policy Sources
filtered_target_ids = t0["target_id"].unique()
t1 = t1[t1["target_id"].isin(filtered_target_ids)]
t2 = t2[t2["target_id"].isin(filtered_target_ids)]


In [33]:
# Register in DuckDB
duckdb.register("tabel0", t0)
duckdb.register("tabel1_exp", t1)
duckdb.register("tabel2_exp", t2)
duckdb.register("tabel3_exp", t3)

In [36]:
result = duckdb.sql("""
SELECT DISTINCT
    tabel1_exp.target_class_name AS target_class,
    tabel2_exp.target_name       AS target,
    tabel3_exp.land_use_name     AS land_use
FROM tabel1_exp
JOIN tabel2_exp
    ON tabel1_exp.target_id = tabel2_exp.target_id
JOIN tabel3_exp
    ON tabel2_exp.land_use_id = tabel3_exp.land_use_id
""").df()


In [ ]:
# Definér labels til Sankey: Target Class → Target → Land uses
source_labels = result['target_class']
middle_labels = result['target']
target_labels = result['land_use']

# Saml og opret unikke labels
all_labels = pd.concat([source_labels, middle_labels, target_labels])
unique_labels = pd.unique(all_labels)
label_to_index = {label: i for i, label in enumerate(unique_labels)}



In [41]:
# Forbindelser
links1 = pd.DataFrame({
    'source': source_labels.map(label_to_index).values,
    'target': target_labels.map(label_to_index).values,
    'value': 1
})


# Saml links, så vi kun har unikke forbindelser med summeret værdi
all_links = (
    links1.groupby(['source', 'target'])
    .size()
    .reset_index(name='value')
)

In [43]:
# Brug Plotlys palette
node_colors = plotly.colors.qualitative.Plotly

# Tildel farver til unikke labels (noder)
color_map = {label: node_colors[i % len(node_colors)] for i, label in enumerate(unique_labels)}

# Liste med farver i samme rækkefølge som unique_labels
node_colors_list = [color_map[label] for label in unique_labels]


# Funktion til at lysne farver (uden brug af gennemsigtighed)
def lighten(hex_color, factor=0.5):
    from plotly.colors import hex_to_rgb
    r, g, b = hex_to_rgb(hex_color)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'rgb({r},{g},{b})'

# Lysnet farve til hver link ud fra source-node
link_colors = [lighten(node_colors_list[src], factor=0.8) for src in all_links['source']]



# Trin 3: Sankey-diagram
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=90,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=list(unique_labels),
        color=node_colors_list
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['value'],
        color=link_colors
    )
)])
fig.update_layout(title_text="Target Class → Land Use", font_size=12, height=3000)
fig.show()

In [ ]:
# Gem som interaktiv HTML-fil
#fig.write_html("sankey_diagram_filtered.html")

# Download i Colab
#from google.colab import files
#files.download("sankey_diagram_filtered.html")

ModuleNotFoundError: No module named 'google'